## 1 - Configurações Gerais

In [1]:
#@markdown Detectando o Ambiente de Execução { display-mode: "form" }
try:
    import google.colab
    EXECUTION_ENV = "colab"
except ImportError:
    try:
        import kaggle_secrets
        EXECUTION_ENV = "kaggle"
    except ImportError:
        EXECUTION_ENV = "local"

print(f"AMBIENTE DE EXECUÇÃO DETECTADO: {EXECUTION_ENV}")

AMBIENTE DE EXECUÇÃO DETECTADO: local


In [2]:
if EXECUTION_ENV in ["colab", "kaggle"]:
    !pip install -q minigrid==3.0.0
    !pip install -q langchain_openai

In [3]:
import os
import sys

if EXECUTION_ENV == "colab":
    repository_path = os.path.join(os.getcwd(), "minigrid_benchmark")

if EXECUTION_ENV == "kaggle":
    repository_path = os.path.join("/kaggle/working/", "minigrid_benchmark")

if EXECUTION_ENV in ["colab", "kaggle"]:
    if not os.path.exists(repository_path):
        !git clone https://github.com/pablo-sampaio/minigrid_benchmark.git
    repository_src_path = os.path.join(repository_path, "src")
    sys.path.append(repository_src_path)
    print(f"Repository src path: {repository_src_path}")

In [ ]:
from langchain_openai import ChatOpenAI

from react_agent import ReActAgent
from wrappers import SYSTEM_PROMPT_WRAPPER_1, SYSTEM_PROMPT_WRAPPER_2d, SYSTEM_PROMPT_WRAPPER_3b, SYSTEM_PROMPT_WRAPPER_3a
from wrappers import OBS_TEMPLATE, OBS_TEMPLATE_ENG

import experiments_util
from experiments_util import run_and_save_experiments, wrapper1, wrapper2_with_numbers, wrapper_local_obs, wrapper_local_obs_noseparation

In [5]:
if EXECUTION_ENV == "colab":
    # Mount Google Drive, to save results there
    from google.colab import drive
    drive.mount('/content/drive')
    results_dir = '/content/drive/My Drive/EAD-Pesquisa-Agentes/_projeto_minigrid/results'

if EXECUTION_ENV == "kaggle":
    results_dir = "/kaggle/working/results"

if EXECUTION_ENV in ["colab", "kaggle"]:
    if not os.path.exists(results_dir):
        os.makedirs(results_dir)
    experiments_util.RESULTS_BASE_DIR = results_dir

In [6]:
OPENAI_KEY_NAME = "OPENAI_PABLO"
OPENAI_API_KEY = None

if EXECUTION_ENV == "colab":
    # just assert that HUGGINGFACE_API_KEY or KF_TOKEN secret is set
    from google.colab import userdata
    OPENAI_API_KEY = userdata.get(OPENAI_KEY_NAME)

if EXECUTION_ENV == "kaggle":
    # para o Kaggle
    from kaggle_secrets import UserSecretsClient
    user_secrets = UserSecretsClient()
    OPENAI_API_KEY = user_secrets.get_secret(OPENAI_KEY_NAME)

if EXECUTION_ENV == "local":
    OPENAI_API_KEY = os.getenv(OPENAI_KEY_NAME)

if not OPENAI_API_KEY:
    raise ValueError(f"{OPENAI_KEY_NAME} environment variable is not set")

## 2 - Configurar o Modelo

In [7]:
MODEL_GPT_5_4 = ChatOpenAI(
    model="gpt-5.4-mini",
    api_key=OPENAI_API_KEY,
)

MODEL_GPT_4_1 = ChatOpenAI(
    model="gpt-4.1-mini",
    api_key=OPENAI_API_KEY,
)

## 3 - Configurações do Experimento

In [ ]:

configs = [
    {
        'name': 'Prompt_1_GPT5.4',
        'agent': ReActAgent(MODEL_GPT_5_4, SYSTEM_PROMPT_WRAPPER_1, OBS_TEMPLATE, verbose=False),
        'wrapper_fn': wrapper1
    },
    {
        'name': 'Prompt_2d_GPT5.4',
        'agent': ReActAgent(MODEL_GPT_5_4, SYSTEM_PROMPT_WRAPPER_2d, OBS_TEMPLATE_ENG, verbose=False),
        'wrapper_fn': wrapper2_with_numbers
    },
    {
        'name': 'Prompt_1_GPT4.1',
        'agent': ReActAgent(MODEL_GPT_4_1, SYSTEM_PROMPT_WRAPPER_1, OBS_TEMPLATE, verbose=False),
        'wrapper_fn': wrapper1
    },
    {
        'name': 'Prompt_2d_GPT4.1',
        'agent': ReActAgent(MODEL_GPT_4_1, SYSTEM_PROMPT_WRAPPER_2d, OBS_TEMPLATE_ENG, verbose=False),
        'wrapper_fn': wrapper2_with_numbers
    },
    {
        'name': 'Prompt_1_GPT5.4_with_history',
        'agent': ReActAgent(MODEL_GPT_5_4, SYSTEM_PROMPT_WRAPPER_1, OBS_TEMPLATE, history_window=5, verbose=False),
        'wrapper_fn': wrapper1
    },
    {
        'name': 'Prompt_2d_GPT5.4_with_history',
        'agent': ReActAgent(MODEL_GPT_5_4, SYSTEM_PROMPT_WRAPPER_2d, OBS_TEMPLATE_ENG, history_window=5, verbose=False),
        'wrapper_fn': wrapper2_with_numbers
    },
        {
        'name': 'Prompt_1_GPT4.1_with_history',
        'agent': ReActAgent(MODEL_GPT_4_1, SYSTEM_PROMPT_WRAPPER_1, OBS_TEMPLATE, history_window=5, verbose=False),
        'wrapper_fn': wrapper1
    },
    {
        'name': 'Prompt_2d_GPT4.1_with_history', # rerunning after bug fix !!!
        'agent': ReActAgent(MODEL_GPT_4_1, SYSTEM_PROMPT_WRAPPER_2d, OBS_TEMPLATE_ENG, history_window=5, verbose=False),
        'wrapper_fn': wrapper2_with_numbers
    },

    # experimento de histórico (para comparar 1, 5 e 9)
    {
        'name': 'Prompt_1_GPT5.4_with_history_3',
        'agent': ReActAgent(MODEL_GPT_5_4, SYSTEM_PROMPT_WRAPPER_1, OBS_TEMPLATE, history_window=3, verbose=False),
        'wrapper_fn': wrapper1
    },
    {
        'name': 'Prompt_2d_GPT5.4_with_history_3',
        'agent': ReActAgent(MODEL_GPT_5_4, SYSTEM_PROMPT_WRAPPER_2d, OBS_TEMPLATE_ENG, history_window=3, verbose=False),
        'wrapper_fn': wrapper2_with_numbers
    },
    {
        'name': 'Prompt_1_GPT5.4_with_history_9',
        'agent': ReActAgent(MODEL_GPT_5_4, SYSTEM_PROMPT_WRAPPER_1, OBS_TEMPLATE, history_window=9, verbose=False),
        'wrapper_fn': wrapper1
    },
    {
        'name': 'Prompt_2d_GPT5.4_with_history_9',
        'agent': ReActAgent(MODEL_GPT_5_4, SYSTEM_PROMPT_WRAPPER_2d, OBS_TEMPLATE_ENG, history_window=9, verbose=False),
        'wrapper_fn': wrapper2_with_numbers
    },

    # experimentos com visão local
    {
        'name': 'Prompt_3local_GPT5.4_without_history',
        'agent': ReActAgent(MODEL_GPT_5_4, SYSTEM_PROMPT_WRAPPER_3b, OBS_TEMPLATE, history_window=1, verbose=False),
        'wrapper_fn': wrapper_local_obs
    },
    {
        'name': 'Prompt_3local_GPT5.4_with_history_3',
        'agent': ReActAgent(MODEL_GPT_5_4, SYSTEM_PROMPT_WRAPPER_3b, OBS_TEMPLATE, history_window=3, verbose=False),
        'wrapper_fn': wrapper_local_obs
    },
    {
        'name': 'Prompt_3local_GPT5.4_with_history_5',
        'agent': ReActAgent(MODEL_GPT_5_4, SYSTEM_PROMPT_WRAPPER_3b, OBS_TEMPLATE, history_window=5, verbose=False),
        'wrapper_fn': wrapper_local_obs
    },
]

## 4 - Execução do Experimento

In [9]:
experiment_name = "2026-05-09_openai"  # se já existir, continua de onde parou

In [10]:
##########################
##  Execute experiments ##
##########################

final_results, filepath = run_and_save_experiments(configs, experiment_name=experiment_name, verbose=True)


Resuming from: d:\Pablo\Documents\Projects\pablo-sampaio\minigrid_benchmark\results\2026-05-09_openai\2026-05-09_openai.json
Results will be saved to: d:\Pablo\Documents\Projects\pablo-sampaio\minigrid_benchmark\results\2026-05-09_openai\2026-05-09_openai.json


Completed Experiment Configurations:   0%|          | 0/15 [00:00<?, ?it/s]

Completed Environments:   0%|          | 0/2 [00:00<?, ?it/s]

Finished Environment Runs::   0%|          | 0/5 [00:00<?, ?it/s]

Finished Environment Runs::   0%|          | 0/5 [00:00<?, ?it/s]

...

Finished Environment Runs::   0%|          | 0/5 [00:00<?, ?it/s]

Finished Environment Runs::   0%|          | 0/5 [00:00<?, ?it/s]

Completed Environments:   0%|          | 0/2 [00:00<?, ?it/s]

Finished Environment Runs::   0%|          | 0/5 [00:00<?, ?it/s]

Finished Environment Runs::   0%|          | 0/5 [00:00<?, ?it/s]

In [11]:
if EXECUTION_ENV in ["colab", "kaggle"]:
    # zip the final results directory
    import shutil

    experiment_result_dir = os.path.dirname(filepath)
    zip_path = os.path.join(experiments_util.RESULTS_BASE_DIR, f"{experiment_name}_results_zip")

    # Creates results_zip.zip containing everything
    shutil.make_archive(zip_path, 'zip', experiment_result_dir)

    print(f"Created: {zip_path}.zip")

In [ ]:
print("Saved results to:", filepath)
print("Now, run notebook \"run_experiments_analysis.ipynb\".")

Now, run notebook "run_experiments_analysis.ipynb".
